Celda 1: Montar Drive, dependencias y crear configs/generacion.yaml

Registra en archivo de configuración todos los hiperparámetros exactos de la Sección 5-C (Qwen/Qwen2.5-3B-Instruct en fp16, temperature=0.3, top_p=0.9, top_k=50, max_new_tokens=450, repetition_penalty=1.05, retries=0).

In [ ]:
import os, json, yaml, random
from collections import Counter
import numpy as np
import torch
from google.colab import drive

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/paperizar'
os.chdir(BASE_DIR)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

!pip install -q transformers accelerate tqdm mlflow pyyaml pandas

assert torch.cuda.is_available(), "Activa GPU T4 en Runtime -> Change runtime type"
print("GPU activa:", torch.cuda.get_device_name(0))

# 1. Guardar configs/generacion.yaml oficial (Sección 5-C y estructura de repo)
gen_cfg = {
    "model_id": "Qwen/Qwen2.5-3B-Instruct",
    "precision": "fp16",
    "do_sample": True,
    "temperature": 0.3,
    "top_p": 0.9,
    "top_k": 50,
    "max_new_tokens": 450,
    "repetition_penalty": 1.05,
    "retries": 0,
    "seed": 42,
    "prompts": {
        "tool": "prompts/P-PAP-v1.txt",
        "baseline": "prompts/P-BASE-v1.txt"
    }
}
with open("configs/generacion.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(gen_cfg, f, sort_keys=False, allow_unicode=True)

# 2. Auditar rápidamente motivos de descarte de pairs.jsonl y cobertura de los 100 abstracts de roundtrip
with open("data/processed/pairs.jsonl", encoding="utf-8") as f:
    pairs = [json.loads(line) for line in f]

kept = [p for p in pairs if p["keep"]]
unique_kept_arxiv = {p["arxiv_id"] for p in kept}
reasons = Counter(p["discard_reason"] for p in pairs if not p["keep"])

print(f"Pares totales: {len(pairs)} | Aprobados (keep=True): {len(kept)}")
print(f"Abstracts únicos de roundtrip con al menos 1 borrador aprobado: {len(unique_kept_arxiv)}/100")
print("Top 5 motivos de descarte en 02_informal:", reasons.most_common(5))

Mounted at /content/drive
GPU activa: Tesla T4
Pares totales: 900 | Aprobados (keep=True): 593
Abstracts únicos de roundtrip con al menos 1 borrador aprobado: 100/100
Top 5 motivos de descarte en 02_informal: [('low_entity_cov(0.88<0.90)', 26), ('low_entity_cov(0.83<0.90)', 15), ('low_entity_cov(0.86<0.90)', 11), ('low_entity_cov(0.67<0.90)', 10), ('low_entity_cov(0.75<0.90)', 9)]


Celda 2: Crear src/paperizar/generate.py (Herramienta P-PAP-v1 y Línea Base P-BASE-v1)Este módulo es el entregable oficial de la Tarea T5. Garantiza:   Cero bucle de regeneración (retries = 0).   Semilla determinista derivada del identificador de cada par.   Si alguno de los 100 abstracts de roundtrip no tuvo ningún borrador con keep=True, procesa adicionalmente su mejor borrador marcado con fallback_100_coverage=True para cumplir simultáneamente el criterio de aceptación de T5 ("Los 100 abstracts de roundtrip procesados sin errores por ambos sistemas") sin mezclar esos casos con el subconjunto estricto keep=True del Paso P4.

In [ ]:
%%writefile src/paperizar/generate.py
import os, re, gc, json, yaml, hashlib
from typing import List, Dict
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

def load_gen_config(cfg_path: str = "configs/generacion.yaml") -> Dict:
    with open(cfg_path, encoding="utf-8") as f:
        return yaml.safe_load(f)

def clean_academic_output(raw: str) -> str:
    txt = raw.strip()
    txt = re.sub(
        r'^(?:Here is[^:]*:|Sure[^:]*:|Abstract:|Academic abstract:|Rewritten text:)\s*',
        '',
        txt,
        flags=re.IGNORECASE
    ).strip()
    txt = txt.strip('"').strip()
    return re.sub(r'\s+', ' ', txt)

def derive_pair_seed(pair_id: str, global_seed: int = 42) -> int:
    h = int(hashlib.md5(f"{global_seed}_{pair_id}".encode("utf-8")).hexdigest()[:8], 16)
    return h % (2**31 - 1)

class PaperizerEngine:
    def __init__(self, cfg_path: str = "configs/generacion.yaml"):
        self.cfg = load_gen_config(cfg_path)
        self.model_id = self.cfg["model_id"]

        with open(self.cfg["prompts"]["tool"], encoding="utf-8") as f:
            self.tpl_tool = f.read().strip()
        with open(self.cfg["prompts"]["baseline"], encoding="utf-8") as f:
            self.tpl_base = f.read().strip()

        self.hash_tool = hashlib.sha256(self.tpl_tool.encode("utf-8")).hexdigest()[:12]
        self.hash_base = hashlib.sha256(self.tpl_base.encode("utf-8")).hexdigest()[:12]

        self.tok = AutoTokenizer.from_pretrained(self.model_id)
        self.tok.padding_side = "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            dtype=torch.float16,
            device_map="auto"
        )
        self.model.eval()

    def generate_batch(self, drafts: List[str], pair_ids: List[str], mode: str = "tool") -> List[Dict]:
        tpl = self.tpl_tool if mode == "tool" else self.tpl_base
        prompt_id = "P-PAP-v1" if mode == "tool" else "P-BASE-v1"
        prompt_hash = self.hash_tool if mode == "tool" else self.hash_base

        formatted = []
        seeds = [derive_pair_seed(pid, self.cfg["seed"]) for pid in pair_ids]
        for d in drafts:
            p_txt = tpl.format(draft=d)
            msgs = [{"role": "user", "content": p_txt}]
            formatted.append(self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))

        inputs = self.tok(
            formatted,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1536
        ).to(self.model.device)

        torch.manual_seed(seeds[0])
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                do_sample=bool(self.cfg["do_sample"]),
                temperature=float(self.cfg["temperature"]),
                top_p=float(self.cfg["top_p"]),
                top_k=int(self.cfg["top_k"]),
                max_new_tokens=int(self.cfg["max_new_tokens"]),
                repetition_penalty=float(self.cfg["repetition_penalty"]),
                pad_token_id=self.tok.pad_token_id,
                eos_token_id=self.tok.eos_token_id,
                use_cache=True
            )

        in_len = inputs["input_ids"].shape[1]
        outputs = []
        for i in range(len(drafts)):
            gen_ids = out[i][in_len:]
            text_out = clean_academic_output(self.tok.decode(gen_ids, skip_special_tokens=True))
            outputs.append({
                "A_prime": text_out,
                "model_id": self.model_id,
                "prompt_id": prompt_id,
                "prompt_hash": prompt_hash,
                "gen_seed": seeds[i]
            })
        return outputs

    def close(self):
        del self.model, self.tok
        gc.collect()
        torch.cuda.empty_cache()

def select_pairs_for_paperization(pairs: List[Dict]) -> List[Dict]:
    """Selecciona todos los keep=True; si algún arxiv_id de los 100 quedó sin representante,
    agrega su borrador de mayor cobertura marcado con fallback_100_coverage=True."""
    selected = []
    covered_arxiv = set()
    for p in pairs:
        if p.get("keep", False):
            p_copy = dict(p)
            p_copy["fallback_100_coverage"] = False
            selected.append(p_copy)
            covered_arxiv.add(p["arxiv_id"])

    by_arxiv = {}
    for p in pairs:
        by_arxiv.setdefault(p["arxiv_id"], []).append(p)

    for aid, group in by_arxiv.items():
        if aid not in covered_arxiv:
            best_p = max(group, key=lambda x: x.get("filter_stats", {}).get("ent_cov", 0.0))
            p_copy = dict(best_p)
            p_copy["fallback_100_coverage"] = True
            selected.append(p_copy)
    return selected

def run_paperization_pipeline(
    input_records: List[Dict],
    out_jsonl: str,
    batch_size: int = 16,
    engine: PaperizerEngine = None
):
    own_engine = False
    if engine is None:
        engine = PaperizerEngine()
        own_engine = True

    done_keys = set()
    if os.path.exists(out_jsonl):
        with open(out_jsonl, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    r = json.loads(line)
                    done_keys.add((r["id"], r["prompt_id"]))
    print(f"[{out_jsonl}] Salidas previas detectadas para reanudar: {len(done_keys)}")

    for mode, pid_tag in [("tool", "P-PAP-v1"), ("baseline", "P-BASE-v1")]:
        pending = [r for r in input_records if (r["id"], pid_tag) not in done_keys]
        if not pending:
            print(f"✔ Sistema {pid_tag} ya está 100% completado en {out_jsonl}.")
            continue

        print(f"\nEjecutando {pid_tag} sobre {len(pending)} borradores (batch_size={batch_size})...")
        with open(out_jsonl, "a", encoding="utf-8") as fout:
            for b_start in tqdm(range(0, len(pending), batch_size), desc=pid_tag):
                batch = pending[b_start:b_start + batch_size]
                drafts = [r["I"] for r in batch]
                pids = [f"{r['id']}_{pid_tag}" for r in batch]
                gen_res = engine.generate_batch(drafts, pids, mode=mode)

                for rec, g in zip(batch, gen_res):
                    out_rec = {
                        **rec,
                        "A_prime": g["A_prime"],
                        "model_id": g["model_id"],
                        "prompt_id": g["prompt_id"],
                        "prompt_hash": g["prompt_hash"],
                        "gen_seed": g["gen_seed"]
                    }
                    fout.write(json.dumps(out_rec, ensure_ascii=False) + "\n")
                fout.flush()

    if own_engine:
        engine.close()

Writing src/paperizar/generate.py


Celda 3: Paso A — Piloto en 10 abstracts comparando Línea Base (P-BASE-v1) frente a Herramienta (P-PAP-v1)

Cumple el flujo de trabajo en Colab de la página 19: correr primero 10 casos e inspeccionar visualmente cómo responden ambos prompts antes de lanzar el lote completo.

In [ ]:
import json, importlib
import src.paperizar.generate as gen_mod
importlib.reload(gen_mod)

with open("data/processed/pairs.jsonl", encoding="utf-8") as f:
    all_pairs = [json.loads(line) for line in f]

selected_pairs = gen_mod.select_pairs_for_paperization(all_pairs)
pilot_10 = selected_pairs[:10]

# Instanciamos el motor una sola vez en VRAM para reusarlo en el piloto y en la corrida completa
engine = gen_mod.PaperizerEngine("configs/generacion.yaml")

gen_mod.run_paperization_pipeline(
    input_records=pilot_10,
    out_jsonl="data/processed/pilot_paperized.jsonl",
    batch_size=10,
    engine=engine
)

# Inspeccionar lado a lado: Borrador I vs Línea Base vs Herramienta
with open("data/processed/pilot_paperized.jsonl", encoding="utf-8") as f:
    p_rows = [json.loads(line) for line in f]

by_id = {}
for r in p_rows:
    by_id.setdefault(r["id"], {})[r["prompt_id"]] = r

sample_id = pilot_10[0]["id"]
s_tool = by_id[sample_id]["P-PAP-v1"]
s_base = by_id[sample_id]["P-BASE-v1"]

print(f"\n=== COMPARACIÓN PILOTO EN PAR: {sample_id} ===")
print("\n1. BORRADOR INFORMAL (I):\n", s_tool["I"])
print("\n2. LÍNEA BASE (P-BASE-v1):\n", s_base["A_prime"])
print("\n3. HERRAMIENTA CON RESTRICCIONES (P-PAP-v1):\n", s_tool["A_prime"])
print("\n4. ORIGINAL ARXIV (A):\n", s_tool["A"])

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

... (5 outputs left out)

P-PAP-v1:   0%|          | 0/1 [00:00<?, ?it/s]


Ejecutando P-BASE-v1 sobre 10 borradores (batch_size=10)...


P-BASE-v1:   0%|          | 0/1 [00:00<?, ?it/s]


=== COMPARACIÓN PILOTO EN PAR: rt_000_G1_L1 ===

1. BORRADOR INFORMAL (I):
 In signal processing, BICM is commonly used in MIMO wireless systems, but there's a trade-off when it comes to MIMO data detection and channel decoding. We're looking at the problem where we only share per-bit soft info, which doesn't account for the correlation between bits in each transmitted vector. This gap in information can really mess with the performance of one-shot receivers. Sure, IDD can handle most of this deficiency, but it's not great for complexity or delay. Here's our fix - we're working on COIL, a new kind of soft-output that factors in per-bit info plus pairwise coupling logits for related bits. By tweaking this into a traditional belief-propagation channel decoder, we're hoping to enable correlation-aware decoding in one shot without relying on IDD. In testing with a MIMO-BICM setup using low-density parity-check codes, our single-pass LMMSE receiver with COIL is beating the standard one-sho

Celda 4: Paso B — Corrida completa sobre roundtrip (y gold_real si ya existe) + Registro en MLflow (T5)

Como aquí hay un solo modelo de 3B en fp16 y cero reintentos, procesar las ~1.186 salidas con batch_size=16 tardará apenas ~20 a 25 minutos (menos de una cuarta parte de lo que tardó 02_informal). Además, si ya creaste gold_real.jsonl (o cuando lo crees más adelante), la misma función lo procesa en 15 segundos.

In [ ]:
import os, json
import mlflow
import pandas as pd

# 1. Procesar el conjunto completo de roundtrip (P-PAP-v1 y P-BASE-v1)
gen_mod.run_paperization_pipeline(
    input_records=selected_pairs,
    out_jsonl="data/processed/paperized_roundtrip.jsonl",
    batch_size=16,
    engine=engine
)

# 2. Si ya existe data/processed/gold_real.jsonl, procesarlo también de una vez
if os.path.exists("data/processed/gold_real.jsonl"):
    with open("data/processed/gold_real.jsonl", encoding="utf-8") as f:
        gold_records = [json.loads(line) for line in f if line.strip()]
    if gold_records:
        print(f"\nDetectados {len(gold_records)} borradores reales en gold_real.jsonl. Paperizando...")
        gen_mod.run_paperization_pipeline(
            input_records=gold_records,
            out_jsonl="data/processed/paperized_gold_real.jsonl",
            batch_size=8,
            engine=engine
        )

# Liberar VRAM al terminar
engine.close()

# 3. Verificar criterio de aceptación de T5 y registrar en MLflow
with open("data/processed/paperized_roundtrip.jsonl", encoding="utf-8") as f:
    out_all = [json.loads(line) for line in f]

df_out = pd.DataFrame(out_all)
cov_by_sys = df_out.groupby("prompt_id")["arxiv_id"].nunique().to_dict()
cnt_by_sys = df_out.groupby("prompt_id")["id"].count().to_dict()
empty_outputs = int((df_out["A_prime"].str.len() < 20).sum())

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("paperizar_generation")
with mlflow.start_run(run_name="qwen2.5_3b_roundtrip_paperization"):
    mlflow.log_params(gen_mod.load_gen_config("configs/generacion.yaml"))
    mlflow.log_metrics({
        "total_outputs": len(df_out),
        "unique_arxiv_P_PAP": cov_by_sys.get("P-PAP-v1", 0),
        "unique_arxiv_P_BASE": cov_by_sys.get("P-BASE-v1", 0),
        "empty_or_broken_outputs": empty_outputs
    })

print("\n=== TAREA 5 CERRADA (04_paperizar.ipynb) ===")
print(f"Salidas generadas por sistema: {cnt_by_sys}")
print(f"Abstracts únicos de roundtrip procesados sin errores (meta = 100): {cov_by_sys}")
print(f"Salidas vacías o truncadas: {empty_outputs}")

[data/processed/paperized_roundtrip.jsonl] Salidas previas detectadas para reanudar: 0

Ejecutando P-PAP-v1 sobre 593 borradores (batch_size=16)...


P-PAP-v1:   0%|          | 0/38 [00:00<?, ?it/s]


Ejecutando P-BASE-v1 sobre 593 borradores (batch_size=16)...


P-BASE-v1:   0%|          | 0/38 [00:00<?, ?it/s]

2026/11/16 17:08:08 INFO mlflow.tracking.fluent: Experiment with name 'paperizar_generation' does not exist. Creating a new experiment.



=== TAREA 5 CERRADA (04_paperizar.ipynb) ===
Salidas generadas por sistema: {'P-BASE-v1': 593, 'P-PAP-v1': 593}
Abstracts únicos de roundtrip procesados sin errores (meta = 100): {'P-BASE-v1': 100, 'P-PAP-v1': 100}
Salidas vacías o truncadas: 0
